In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("phase2-preprocess").getOrCreate()
tx = spark.read.parquet("/home/jovyan/work/data/processed/train_transaction")
total = tx.count()
print(f"loaded {total:,} rows, {len(tx.columns)} cols")

loaded 590,540 rows, 394 cols


In [2]:
cols = tx.columns
null_rates = {c: (tx.filter(F.col(c).isNull()).count()) / total for c in cols}   # per-column missing %
to_drop_missing = [c for c, r in null_rates.items() if r > 0.80]
tx = tx.drop(*to_drop_missing)
print(f"dropped {len(to_drop_missing)} columns for >80% missingness → surviving {len(tx.columns)} cols")

dropped 55 columns for >80% missingness → surviving 339 cols


In [4]:
from pyspark.sql.functions import corr

v_cols = [c for c in tx.columns if c.startswith("V")]
print(f"checking {len(v_cols)} V-columns for near-duplicate behavior…")

# scale-snapshot trick: represent each V-column as a tiny numeric fingerprint vector in a single pass
v_sample_df = tx.sample(False, 0.05, seed=42).toPandas()   # pull a 5% sample once (~29k rows × ~340 vb cols = manageable memory)
import numpy as np
V = v_sample_df[v_cols].fillna(0).values.T       # shape: (v_cols, rows) → compute col-col correlation as ONE matrix op
R = np.corrcoef(V)                                # ~340 × 340 matrix = instant

# greedy dedup on the matrix (drop the second of every >0.98 pair)
drop_v, keep_v = [], []
for i in range(len(v_cols)):
    if v_cols[i] in drop_v: 
        continue
    keep_v.append(v_cols[i])
    for j in range(i+1, len(v_cols)):
        if abs(R[i, j]) > 0.98:
            drop_v.append(v_cols[j])

tx = tx.drop(*drop_v)
print(f"V-columns surviving deduplication: {len(keep_v)} of {len(v_cols)}")

checking 292 V-columns for near-duplicate behavior…
V-columns surviving deduplication: 243 of 292


In [5]:
tx.write.mode("overwrite").parquet("/home/jovyan/work/data/processed/train_transaction_clean")
print(f"saved clean data: {tx.count():,} rows × {len(tx.columns)} cols")

saved clean data: 590,540 rows × 290 cols
